In [13]:
!pip install pandas

In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("shopey.db")
cursor = conn.cursor()

print("Database created")

Database created


In [16]:
cursor.executescript("""

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name VARCHAR(100) NOT NULL,
    last_name VARCHAR(100) NOT NULL,
    email VARCHAR(255) UNIQUE NOT NULL,
    phone VARCHAR(20),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name VARCHAR(100) NOT NULL,
    description TEXT
);

CREATE TABLE vendors (
    vendor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    vendor_name VARCHAR(150) NOT NULL,
    contact_email VARCHAR(255)
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name VARCHAR(200) NOT NULL,
    category_id INTEGER,
    vendor_id INTEGER,
    unit_price NUMERIC(10,2) NOT NULL,
    stock_qty INTEGER DEFAULT 0
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    status VARCHAR(20) DEFAULT 'Pending'
);

CREATE TABLE order_lines (
    line_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price NUMERIC(10,2) NOT NULL
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER UNIQUE NOT NULL,
    payment_date TIMESTAMP,
    method VARCHAR(50),
    amount NUMERIC(10,2) NOT NULL,
    status VARCHAR(20) DEFAULT 'Pending'
);

""")

conn.commit()

In [17]:
cursor.executescript("""

-- Customers
INSERT INTO customers
(first_name, last_name, email)
VALUES
('Alice', 'Smith', 'alice@test.com'),
('Bob', 'Johnson', 'bob@test.com'),
('Carol', 'White', 'carol@test.com');

-- Categories
INSERT INTO categories
(category_name, description)
VALUES
('Electronics', 'Electronic products'),
('Books', 'Books and study material');

-- Vendors
INSERT INTO vendors
(vendor_name, contact_email)
VALUES
('TechVendor', 'tech@test.com'),
('BookVendor', 'books@test.com');

-- Products
INSERT INTO products
(product_name, category_id, vendor_id, unit_price, stock_qty)
VALUES
('Laptop', 1, 1, 50000, 10),
('Python Book', 2, 2, 500, 100);

-- Orders
INSERT INTO orders
(customer_id, status)
VALUES
(1, 'Delivered'),
(1, 'Delivered'),
(2, 'Delivered'),
(3, 'Delivered');

-- Order Lines
INSERT INTO order_lines
(order_id, product_id, quantity, unit_price)
VALUES
(1, 1, 1, 50000),
(2, 2, 3, 500),
(3, 1, 1, 50000),
(4, 2, 2, 500);

-- Payments
INSERT INTO payments
(order_id, payment_date, method, amount, status)
VALUES
(1, CURRENT_TIMESTAMP, 'Card', 50000, 'Paid'),
(2, CURRENT_TIMESTAMP, 'Wallet', 1500, 'Paid'),
(3, CURRENT_TIMESTAMP, 'Card', 50000, 'Paid'),
(4, CURRENT_TIMESTAMP, 'PayPal', 1000, 'Paid');

""")

conn.commit()

print("Sample data inserted successfully!")

Sample data inserted successfully!


In [18]:
query = """

-- Get customer purchase summary and rank customers by spending
-- Combine first name and last name into a single column
-- Count unique orders placed by each customer
-- Calculate total amount spent by customer (quantity purchased × unit_price)
-- Rank customers based on total spending (highest spender gets rank 1)

SELECT
    c.first_name || ' ' || c.last_name AS customer_name,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(ol.quantity * ol.unit_price) AS total_spent,
    RANK() OVER (ORDER BY SUM(ol.quantity * ol.unit_price) DESC) AS customer_rank
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_lines ol
    ON o.order_id = ol.order_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name
ORDER BY customer_rank;

"""

In [19]:
df = pd.read_sql_query(query, conn)

df

,customer_name,total_orders,total_spent,customer_rank
0,Alice Smith,2,51500,1
1,Bob Johnson,1,50000,2
2,Carol White,1,1000,3
